<a href="https://colab.research.google.com/github/Aswitha2114/MY-Projects/blob/main/clinic_and_patient_care.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#TASK 1
import pandas as pd
from datetime import datetime
inventory_data = pd.DataFrame({
    "Item": ["Paracetamol", "Gloves", "Syringes", "IV Fluids"],
    "Opening_Stock": [50, 6, 150, 12],
    "Used_Today": [25, 3, 70, 4],
    "Expiry_Date": ["2026-02-15", "2026-01-25", "2027-01-01", "2026-01-10"]
})

thresholds = pd.DataFrame({
    "Item": ["Paracetamol", "Gloves", "Syringes", "IV Fluids"],
    "Min_Stock": [30, 5, 100, 10]
})

df = inventory_data.merge(thresholds, on="Item")
df["Closing_Stock"] = df["Opening_Stock"] - df["Used_Today"]
df["Expiry_Date"] = pd.to_datetime(df["Expiry_Date"])
today = pd.to_datetime(datetime.today().date())

def stock_status(row):
    if row["Closing_Stock"] == 0:
        return "Critical"
    elif row["Closing_Stock"] < row["Min_Stock"]:
        return "Low"
    else:
        return "OK"

df["Stock_Status"] = df.apply(stock_status, axis=1)

df["Days_To_Expiry"] = (df["Expiry_Date"] - today).dt.days
df["Expiry_Status"] = df["Days_To_Expiry"].apply(
    lambda x: "Expiring Soon" if x <= 30 else "OK"
)

df["Action_Required"] = df.apply(
    lambda row: "Yes" if row["Stock_Status"] != "OK" or row["Expiry_Status"] != "OK" else "No",
    axis=1
)

final_report = df[[
    "Item",
    "Opening_Stock",
    "Used_Today",
    "Closing_Stock",
    "Min_Stock",
    "Stock_Status",
    "Expiry_Date",
    "Expiry_Status",
    "Action_Required"
]]

final_report

,Item,Opening_Stock,Used_Today,Closing_Stock,Min_Stock,Stock_Status,Expiry_Date,Expiry_Status,Action_Required
0,Paracetamol,50,25,25,30,Low,2026-02-15,OK,Yes
1,Gloves,6,3,3,5,Low,2026-01-25,Expiring Soon,Yes
2,Syringes,150,70,80,100,Low,2027-01-01,OK,Yes
3,IV Fluids,12,4,8,10,Low,2026-01-10,Expiring Soon,Yes


In [ ]:
#Task 2
import pandas as pd
from datetime import datetime

hms_data = pd.DataFrame({
    "Patient_Name": ["Ramesh K", "Sita P", "Arjun M"],
    "Phone": ["9XXXXXXX1", "9XXXXXXX2", "9XXXXXXX3"],
    "Visit_Type": ["OPD", "Procedure", "OPD"],
    "Treatment": ["Viral Fever Meds", "Minor Skin Procedure", "Gastric Medication"],
    "Follow_Up_Date": ["2026-08-05", "2026-08-07", None]
})

patient_questions = pd.DataFrame({
    "Patient_Name": ["Lakshmi"],
    "Phone": ["9XXXXXXX4"],
    "Question": ["Is itching normal after procedure?"],
    "Urgency": ["Routine"]
})

def classify_message(row):
    if row["Visit_Type"] == "Procedure":
        return "Post-Procedure Care"
    elif row["Visit_Type"] == "OPD" and pd.notnull(row["Follow_Up_Date"]):
        return "Follow-up Reminder"
    else:
        return "Custom Instruction"

hms_data["Message_Type"] = hms_data.apply(classify_message, axis=1)
def approval_needed(msg_type):
    if msg_type in ["Follow-up Reminder", "Post-Procedure Care"]:
        return "Not Required"
    else:
        return "Required"

hms_data["Doctor_Approval"] = hms_data["Message_Type"].apply(approval_needed)

def generate_message(row):
    if row["Message_Type"] == "Follow-up Reminder":
        return f"Please visit the clinic on {row['Follow_Up_Date']} for your follow-up."
    elif row["Message_Type"] == "Post-Procedure Care":
        return "Mild swelling is normal. Please follow the care instructions provided."
    else:
        return None

hms_data["Message_Text"] = hms_data.apply(generate_message, axis=1)

def initial_status(row):
    if row["Doctor_Approval"] == "Required":
        return "Waiting"
    else:
        return "Pending"

hms_data["Status"] = hms_data.apply(initial_status, axis=1)

care_control = hms_data[[
    "Patient_Name",
    "Phone",
    "Visit_Type",
    "Message_Type",
    "Message_Text",
    "Doctor_Approval",
    "Status"
]]

patient_questions["Message_Type"] = "Patient Question Response"
patient_questions["Doctor_Approval"] = "Required"
patient_questions["Message_Text"] = None
patient_questions["Status"] = "Waiting"
final_model_view = pd.concat([
    care_control,
    patient_questions[[
        "Patient_Name",
        "Phone",
        "Message_Type",
        "Message_Text",
        "Doctor_Approval",
        "Status"
    ]]
], ignore_index=True)

final_model_view


,Patient_Name,Phone,Visit_Type,Message_Type,Message_Text,Doctor_Approval,Status
0,Ramesh K,9XXXXXXX1,OPD,Follow-up Reminder,Please visit the clinic on 2026-08-05 for your...,Not Required,Pending
1,Sita P,9XXXXXXX2,Procedure,Post-Procedure Care,Mild swelling is normal. Please follow the car...,Not Required,Pending
2,Arjun M,9XXXXXXX3,OPD,Custom Instruction,None,Required,Waiting
3,Lakshmi,9XXXXXXX4,NaN,Patient Question Response,None,Required,Waiting


from matplotlib import pyplot as plt
import seaborn as sns
final_model_view.groupby('Patient_Name').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
final_model_view.groupby('Phone').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
final_model_view.groupby('Visit_Type').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
final_model_view.groupby('Message_Type').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['Phone'].value_counts()
    for x_label, grp in final_model_view.groupby('Patient_Name')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Patient_Name')
_ = plt.ylabel('Phone')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['Visit_Type'].value_counts()
    for x_label, grp in final_model_view.groupby('Phone')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Phone')
_ = plt.ylabel('Visit_Type')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['Message_Type'].value_counts()
    for x_label, grp in final_model_view.groupby('Visit_Type')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Visit_Type')
_ = plt.ylabel('Message_Type')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['Message_Text'].value_counts()
    for x_label, grp in final_model_view.groupby('Message_Type')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Message_Type')
_ = plt.ylabel('Message_Text')